In [1]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-

'''
Created on 2024-09-02
Last modified on 2024-09-06
@author: Juan Enrique López
@description: Jupyter Notebook creado para obtener los ficheros .md con la clasificación por actor (grupo) y táctica de cada riesgo.

'''

'\nCreated on 2024-09-02\nLast modified on 2024-09-06\n@author: Juan Enrique López\n@description: Jupyter Notebook creado para obtener los ficheros .md con la clasificación por actor (grupo) y táctica de cada riesgo.\n\n'

**IMPORTANTE - REQUERIMIENTOS PREVIOS IMPRESCINDIBLES**

Archivos necesarios:

- Requiere disponer de los outputs (csv) que contienen la información relativa a las propiedades Mitre. Estos se obtienen tras ejecutar el nb *[stix2]_mitre_relationships.ipynb* perteneciente al módulo **mitre_relationships**.

- Requiere disponer de las reglas de detección clasificadas por dominio Mitre y origen. Esta clasificación de las reglas de detección es realizada por el nb *get_rules_and_classify_by_ttp.ipynb* que se encuentra en este mismo módulo **get_rules_and_classify_by_ttp**.


Configuración necesaria:

- Disponer de 7z en las variables de entorno del sistema (PATH).

Inputs:

- Se requiere que, el o los archivos input estén guardados en la carpeta *input_actor*. Esta se ubica en la misma carpeta que este jupyter notebook.

In [2]:
import os
import pandas as pd
import shutil
import ast
from stix2 import Filter, MemoryStore
import numpy as np
import subprocess

##### **Funciones**

In [3]:
def check_and_create_folder(path):
    '''
    Función que verifica si una ruta existe y la crea si es necesario.
    '''
    try:
        if not os.path.exists(path):
            os.makedirs(path)
            print(f"Se ha creado la ruta: {path}")
        else:
            print(f"La ruta ya existe: {path}")
    except Exception as e:
        print(f"Error al crear la ruta {path}: {e}")

In [4]:
def list_files_in_directory(directory_path):
    '''
    Función encargada de listar solo los archivos CSV en el directorio facilitado.
    '''
    try:
        # Listar todos los archivos en la ruta dada
        files = os.listdir(directory_path)
        
        # Filtrar para obtener solo archivos CSV
        csv_files = [f for f in files if f.endswith('.csv') and os.path.isfile(os.path.join(directory_path, f))]
        
        if not csv_files:
            raise ValueError(f"No se encontraron archivos CSV en la carpeta '{directory_path}'.")
        
        # Mostrar el número de archivos CSV y sus rutas
        print(f"Se encontraron {len(csv_files)} archivo(s) CSV en la carpeta '{directory_path}':")
        for csv_file in csv_files:
            print(os.path.join(directory_path, csv_file))
        
        # Retornar la lista de rutas completas de los archivos CSV
        return [os.path.join(directory_path, f) for f in csv_files]
    
    except FileNotFoundError:
        print(f"La ruta '{directory_path}' no existe.")
        return None
    
    except ValueError as ve:
        return None

In [5]:
def clean_input_df(df):
    '''
    Función encargada de realizar ciertas transformaciones en el fichero input.
    '''
    df.columns = df.columns.str.lower().str.replace(' ', '_')
    df = df.rename(columns={'mitre_category_name': 'tactic', 'technique_name': 'technique', 'actors': 'group', 'malware': 'software'})
    df['tactic'] = df['tactic'].str.lower()
    df['group'] = df['group'].str.lower()
    df = df.drop(columns=['sub-technique_ids', 'sub-technique_names', 'usage_count', 'campaigns'])
    df = df.explode('group').reset_index(drop=True)
    df['group'] = df['group'].str.split(',')
    df['group'] = df['group'].apply(lambda x: [item.strip() for item in x] if isinstance(x, list) else x)
    df = df.explode('group').reset_index(drop=True)
    return df

In [6]:
def get_rules_df(folder_source, list_ttp, path_rules):
    '''
     Función cuyo cometido es crear un dataframe que contenga el id de la ttp, nombre de la regla, origen y ruta.
    '''
    # Crear un DataFrame vacío con las columnas 'ttp', 'rule', 'source', 'path'
    df = pd.DataFrame(columns=['technique ID', 'rule', 'source', 'path'])
    # Iterar sobre las carpetas de origen
    for source in folder_source:
        # Iterar sobre la lista de TTP
        for ttp in list_ttp:
            # Construir la ruta a las reglas
            rules = os.path.join(path_rules, source, ttp)
            # Comprobar si la ruta existe
            if os.path.exists(rules):
                # Listar los archivos en la carpeta de reglas
                rule_list = os.listdir(rules)
                # Crear las rutas completas para cada regla
                full_paths = [os.path.join(rules, rule) for rule in rule_list]
                # Crear un DataFrame temporal con la información actual
                temp_df = pd.DataFrame({
                    'technique ID': [ttp] * len(rule_list),
                    'rule': rule_list,
                    'source': [source] * len(rule_list),
                    'path': full_paths
                })
                # Concatenar el DataFrame temporal con el principal
                df = pd.concat([df, temp_df], ignore_index=True)

    # Ordenar el DataFrame por el campo 'ttp' en orden descendente y resetear el índice
    return df.sort_values(by='technique ID', ascending=False).reset_index(drop=True)

In [7]:
def enable_long_path(path):
    '''
    Complemento de la funcion copy_files() que permite el uso de rutas superiores a los 260 caracteres.
    '''
    if os.name == 'nt':
        if not path.startswith('\\\\?\\'):
            path = '\\\\?\\' + os.path.abspath(path)
    return path

In [8]:
def shorten_filename_if_needed(destination_path, max_path_length=260, max_filename_length=30):
    '''
    Complemento de la funcion copy_files() para reducir el nombre del archivo si es necesario.
    '''
    if len(destination_path) > max_path_length:
        # Separar la ruta y el nombre del archivo
        dir_path, filename = os.path.split(destination_path)
        
        # Separar nombre y extensión
        file_name, file_ext = os.path.splitext(filename)
        
        # Si el nombre del archivo sin extensión es más largo que el límite, recortarlo
        if len(file_name) > max_filename_length:
            file_name = file_name[:max_filename_length]  # Tomar los primeros 30 caracteres
        
        # Crear el nuevo destino con el nombre reducido
        new_filename = file_name + file_ext
        destination_path = os.path.join(dir_path, new_filename)

    return destination_path

In [9]:
def copy_files(df, path_output, optional_folder, group):
    '''
    Función encargada de recorrer el dataframe y copiar los archivos a una nueva estructura de carpetas.
    '''
    for _, row in df.iterrows():
        # Obtener el ttp, rule y la ruta completa del archivo
        ttp = row['technique ID']
        rule = row['rule']
        source_path = row['path']

        # Crear la ruta destino en la estructura output/[technique ID]/[rule]
        destination_dir = os.path.join(path_output, optional_folder, group, ttp)
        destination_path = os.path.join(destination_dir, rule)

        # Reducir el nombre del archivo si la ruta es demasiado larga
        destination_path = shorten_filename_if_needed(destination_path)

        # Crear las carpetas si no existen
        os.makedirs(destination_dir, exist_ok=True)

        # Habilitar rutas largas en origen y destino
        long_source_path = enable_long_path(source_path)
        long_destination_path = enable_long_path(destination_path)

        # Copiar el archivo al destino usando rutas largas
        shutil.copy(long_source_path, long_destination_path)

In [10]:
def get_data_from_techniques(matrix):
    '''
    Función cuyo cometido es leer el archivo csv de técnicas generado por el módulo mitre_relationships y el nb [stix2]_mitre_relationships.ipynb. El archivo consultado se corresponderá con la matriz facilitada en el argumento obligatorio.
    '''
    file_name = f'[MITRE]_{matrix}_techniques.csv'
    path_file = os.path.join(os.path.dirname(os.getcwd()), 'mitre_relationships', 'outputs\stix2', matrix, r'information\techniques',file_name)
    information_df = pd.read_csv(path_file, sep=';', quotechar='"')
    return information_df

In [11]:
def get_data_from_relation_techniques_tactics(matrix):
    '''
    Función cuyo cometido es leer el archivo csv de técnicas generado por el módulo mitre_relationships y el nb [stix2]_mitre_relationships.ipynb. El archivo consultado se corresponderá con la matriz facilitada en el argumento obligatorio.
    '''
    file_name = f'[MITRE]_{matrix}_technique_tactics_1N.csv'
    path_file = os.path.join(os.path.dirname(os.getcwd()), 'mitre_relationships', 'outputs\stix2', matrix, r'relations\techniques_tactics',file_name)
    information_df = pd.read_csv(path_file, sep=';', quotechar='"')
    return information_df

In [12]:
def get_data_from_relation_techniques_software(matrix):
    '''
    Función cuyo cometido es leer el archivo csv de técnicas generado por el módulo mitre_relationships y el nb [stix2]_mitre_relationships.ipynb. El archivo consultado se corresponderá con la matriz facilitada en el argumento obligatorio.
    '''
    file_name = f'[MITRE]_{matrix}_technique_software_1N.csv'
    path_file = os.path.join(os.path.dirname(os.getcwd()), 'mitre_relationships', 'outputs\stix2', matrix, r'relations\techniques_software',file_name)
    information_df = pd.read_csv(path_file, sep=';', quotechar='"')
    return information_df

In [13]:
def get_data_from_relation_techniques_platforms(matrix):
    '''
    Función cuyo cometido es leer el archivo csv de técnicas generado por el módulo mitre_relationships y el nb [stix2]_mitre_relationships.ipynb. El archivo consultado se corresponderá con la matriz facilitada en el argumento obligatorio.
    '''
    file_name = f'[MITRE]_{matrix}_technique_platforms_1N.csv'
    path_file = os.path.join(os.path.dirname(os.getcwd()), 'mitre_relationships', 'outputs\stix2', matrix, r'relations\techniques_platforms',file_name)
    information_df = pd.read_csv(path_file, sep=';', quotechar='"')
    return information_df


In [14]:
def get_data_from_relation_technique_groups(matrix):
    '''
    Función cuyo cometido es leer el archivo csv de técnicas generado por el módulo mitre_relationships y el nb [stix2]_mitre_relationships.ipynb. El archivo consultado se corresponderá con la matriz facilitada en el argumento obligatorio.
    '''
    file_name = f'[MITRE]_{matrix}_technique_groups_1N.csv'
    path_file = os.path.join(os.path.dirname(os.getcwd()), 'mitre_relationships', 'outputs\stix2', matrix, r'relations\techniques_groups',file_name)
    information_df = pd.read_csv(path_file, sep=';', quotechar='"')
    return information_df

In [15]:
def get_information_data_from_groups(matrix):
    '''
    Función encargada de retornar el df con la información relativa a los grupos MITRE.
    '''
    file_name = f'[MITRE]_{matrix}_groups.csv'
    path_file = os.path.join(os.path.dirname(os.getcwd()), 'mitre_relationships', 'outputs\stix2', matrix, r'information\groups',file_name)
    information_df = pd.read_csv(path_file, sep=';', quotechar='"')
    return information_df

In [16]:
def get_information_data_from_tactics(matrix):
    '''
    Función encargada de retornar el df con la información relativa a las tácticas MITRE.
    '''
    file_name = f'[MITRE]_{matrix}_tactics.csv'
    path_file = os.path.join(os.path.dirname(os.getcwd()), 'mitre_relationships', 'outputs\stix2', matrix, r'information\tactics',file_name)
    information_df = pd.read_csv(path_file, sep=';', quotechar='"')
    return information_df

In [17]:
def get_data_from_relation_techniques_datasources(matrix):
    '''
    Función cuyo cometido es leer el archivo csv de técnicas generado por el módulo mitre_relationships y el nb [stix2]_mitre_relationships.ipynb. El archivo consultado se corresponderá con la matriz facilitada en el argumento obligatorio.
    '''
    file_name = f'[MITRE]_{matrix}_technique_datasources_1N.csv'
    path_file = os.path.join(os.path.dirname(os.getcwd()), 'mitre_relationships', 'outputs\stix2', matrix, r'relations\techniques_datasources',file_name)
    information_df = pd.read_csv(path_file, sep=';', quotechar='"')
    return information_df

In [18]:
def get_group_tactic_information(matrix):
    '''
    Función encargada de obtener y formatear las tablas informativas de grupos y tácticas.
    '''
    groups_df = get_information_data_from_groups(matrix)
    groups_df['group_aliases'] = groups_df['group_aliases'].str.lower()
    groups_df['group_aliases'] = groups_df['group_aliases'].apply(ast.literal_eval)
    groups_df = groups_df.explode('group_aliases').reset_index(drop=True)
    groups_df = groups_df.drop(columns=['matrix_domains', 'group_deprecated', 'group_revoked'])
    groups_df = groups_df.rename(columns={'group': 'group_main_name', 'group_aliases': 'group'})

    tactics_df = get_information_data_from_tactics(matrix)
    tactics_df = tactics_df.drop(columns=['matrix_domains'])
    tactics_df['tactic'] = tactics_df['tactic'].str.lower()

    return groups_df, tactics_df

In [19]:
def format_cols_for_md(column):
    '''
    Función para aplicar los reemplazos que permitan en el .md relacionar elementos.
    '''
    if column.name == 'technique_ID':
        column = column.apply(lambda x: f'[[{x}]]')
    elif column.dtype == "object":  # Solo aplica a columnas de tipo string
        column = column.str.replace("'", '', regex=False)
        column = column.str.replace("[", '[[', regex=False)
        column = column.str.replace("]", ']]', regex=False)
        column = column.str.replace(", ", ']] [[', regex=False)
    return column

In [20]:
def replace_chars(column):
    '''
    Función para eliminar caracteres extraños en los str de una columna de dataframe.
    '''
    if column.dtype == "object":  # Verifica que la columna sea de tipo string
        column = column.str.replace("['", '', regex=False)
        column = column.str.replace("']", '', regex=False)
        column = column.str.replace("'", '', regex=False)
    return column

In [21]:

def generate_markdown(temp_union_df, df_techniques, resume_rules_df, df_rules, folder_path, group, rules_technique_group, optional_folder=''):
    '''
    Función prinicpal de generación de archivos .md. Se encargará de construir los archivos .md con la información relativa a los inputs e información complementaria requerida.
    '''
    # Asegúrate de que la carpeta exista
    os.makedirs(folder_path, exist_ok=True)
    
    # Construir la ruta completa del archivo principal
    main_filename = f'riesgos_{optional_folder}_{group.upper()}.md'
    main_file_path = os.path.join(folder_path, main_filename)
    resume_techniques_tactic = temp_union_df.groupby(['group', 'tactic'])['technique_id'].nunique().reset_index(name='techniques')


    with open(main_file_path, 'w') as main_file:
        # Escribir el contenido inicial hasta el DataFrame df_techniques
        main_file.write("---\n")
        main_file.write("Risk: true\n")
        main_file.write("Technique: true\n")
        main_file.write("Tactic: true\n")
        main_file.write("Platform: true\n")
        main_file.write("Data source: true\n")
        main_file.write("Group: true\n")
        main_file.write("Software: true\n")
        main_file.write("---\n\n")

        main_file.write(f"**Archivo padre:**\n")
        main_file.write(f"[[riesgos_{optional_folder}.md]]\n\n")
        # Agregar enlaces a los archivos individuales para cada técnica
        main_file.write(f"**Archivos relacionados con el {optional_folder} y el grupo [[{group.upper()}]] por técnica:**\n\n")
        for _, row in temp_union_df.iterrows():
            technique_id = row['technique_id'].strip('[]')
            individual_filename = f"riesgos_{optional_folder}_{group.upper()}_{technique_id}.md"
            main_file.write(f"[[{individual_filename}]]\n")
        main_file.write("\n\n")

        main_file.write(f"### Resumen técnicas por táctica para el grupo [[{group.upper()}]]:\n\n")
        main_file.write("| group | tactic | techniques |\n")
        main_file.write("| ------------ | ------ | ---- |\n")
        for _, row in resume_techniques_tactic.iterrows():
            main_file.write(f"| {row['group']} | {row['tactic']} | {row['techniques']} |\n")
        main_file.write("\n\n")

        main_file.write(f"### Resumen reglas de detección por técnica para el grupo [[{group.upper()}]]:\n\n")
        main_file.write("| technique | rules |\n")
        main_file.write("| ------------ | ------ |\n")
        for _, row in rules_technique_group.iterrows():
            main_file.write(f"| {row['technique ID']} | {row['rules']}|\n")
        main_file.write("\n\n")
        main_file.close()

    # print(f"Archivo '{main_filename}' creado en la carpeta '{folder_path}' con éxito.")

    # Crear archivos individuales para cada fila en df_techniques
    for index, row in df_techniques.iterrows():
        
        technique_id = row['technique ID'].strip('[]')
        df_rules_technique = df_rules[df_rules['technique ID']==technique_id]
        union_group_technique_df  = temp_union_df[temp_union_df['technique_id']==technique_id]
        original_tactic = union_group_technique_df['tactic'].iloc[0]
        resume_rules_technique_df = resume_rules_df[resume_rules_df['technique ID']==technique_id]
        individual_filename = f"riesgos_{optional_folder}_{group.upper()}_{technique_id}.md"  # El nombre del archivo basado en technique ID
        individual_file_path = os.path.join(folder_path, group.upper(), individual_filename)
        
        with open(individual_file_path, 'w') as individual_file:
            # Escribir el contenido del archivo basado en la fila actual
            individual_file.write("---\n")
            individual_file.write("Risk: true\n")
            individual_file.write("Risk relations: true\n")
            individual_file.write("Risk relations technique: true\n")
            individual_file.write("---\n\n")

            individual_file.write(f"**Archivo padre:**\n")
            individual_file.write(f"[[riesgos_{optional_folder}_{group.upper()}.md]]\n\n")
            
            individual_file.write(f"#### Táctica asignada a la técnica en el archivo original: {original_tactic}\n\n")

            # Información de la técnica
            individual_file.write(f"#### Información relacional de MITRE: \n\n")
            individual_file.write(f"#### {technique_id}\n\n")
            individual_file.write(f"**technique ID**\n{row['technique ID']}\n\n")
            individual_file.write(f"**technique**\n{row['technique']}\n\n")
            individual_file.write(f"**technique url**\n{row['technique url']}\n\n")
            individual_file.write(f"**technique description**\n{row['technique description']}\n\n")
            individual_file.write(f"**technique deprecated**\n{row['technique deprecated']}\n\n")
            individual_file.write(f"**technique revoked**\n{row['technique revoked']}\n\n")
            individual_file.write(f"**matrix domains**\n{row['matrix domains']}\n\n")
            individual_file.write(f"**tactic ID**\n{row['tactic ID']}\n\n")
            individual_file.write(f"**tactic**\n{row['tactic']}\n\n")
            individual_file.write(f"**software ID**\n{row['software ID']}\n\n")
            individual_file.write(f"**software**\n{row['software']}\n\n")
            individual_file.write(f"**platform**\n{row['platform']}\n\n")
            individual_file.write(f"**group ID**\n{row['group ID']}\n\n")
            individual_file.write(f"**group**\n{row['group']}\n\n")
            individual_file.write(f"**data source ID**\n{row['data source ID']}\n\n")
            individual_file.write(f"**data source**\n{row['data source']}\n\n")

            # pendiente confirmar
            individual_file.write(f"### Resumen reglas de detección disponibles para la técnica [[{technique_id}]]:\n\n")
            individual_file.write("| technique ID | source | rule |\n")
            individual_file.write("| ------------ | ------ | ---- |\n")
            for _, row in resume_rules_technique_df.iterrows():
                individual_file.write(f"| {row['technique ID']} | {row['source']} | {row['rules']} |\n")
            individual_file.write("\n\n")
            
            individual_file.write("### Desglose reglas de detección disponibles:\n\n")
            individual_file.write("| technique ID | source | rules |\n")
            individual_file.write("| ------------ | ------ | ---- |\n")
            for _, row in df_rules_technique.iterrows():
                individual_file.write(f"| {row['technique ID']} | {row['source']} | {row['rule']} |\n")
            individual_file.write("\n\n")
            
            # Añadir una línea en blanco para separar secciones
            individual_file.write("\n\n")
            individual_file.close()
        
        # print(f"Archivo '{individual_filename}' creado en la carpeta '{folder_path}' con éxito.")
    
    print(f"Proceso de generacion de archivos .md finalizado correctamente!")

In [22]:
def info_techniques(results_df, mitre_domain):
    '''
    Función encargada de filtrar mediante el df facilitado y formatear la tabla de información de técnicas. 
    '''
    techniques_df = get_data_from_techniques(mitre_domain)
    techniques_df = techniques_df[techniques_df['technique_ID'].isin(results_df['technique ID'].unique().tolist())]
    techniques_df = techniques_df.sort_values(by='technique_ID', ascending=False).reset_index(drop=True)
    
    column_list = ['technique_ID', 'tactic_ID', 'software_ID', 'platform', 'group_ID', 'data_source_ID']
    for col in techniques_df.columns:
        if col in column_list:
            techniques_df[col] = techniques_df[col].apply(lambda x: f'[[{x}]]')
        elif techniques_df[col].dtype == "object":  # Solo aplicar si es string
            techniques_df[col] = replace_chars(techniques_df[col])
    return techniques_df

In [23]:
def info_techniques_tactics(results_df, mitre_domain):
    '''
    Función encargada de filtrar mediante el df facilitado y formatear la tabla de información de tácticas. 
    '''
    techniques_tactics_df = get_data_from_relation_techniques_tactics(mitre_domain)
    techniques_tactics_df = techniques_tactics_df[techniques_tactics_df['technique_ID'].isin(results_df['technique ID'].unique().tolist())]
    techniques_tactics_df = techniques_tactics_df.sort_values(by='technique_ID', ascending=False).reset_index(drop=True)
    column_list = ['technique_ID','tactic_ID','software_ID','platform','group_ID','data_source_ID']
    for col in techniques_tactics_df.columns:
        if col in column_list:
            techniques_tactics_df[col] = format_cols_for_md(techniques_tactics_df[col])
        else:
           techniques_tactics_df[col] =  replace_chars(techniques_tactics_df[col])
    return techniques_tactics_df

In [24]:
def info_techniques_software(results_df, mitre_domain):
    '''
    Función encargada de filtrar mediante el df facilitado y formatear la tabla de información de software. 
    '''
    techniques_software_df = get_data_from_relation_techniques_software(mitre_domain)
    techniques_software_df = techniques_software_df[techniques_software_df['technique_ID'].isin(results_df['technique ID'].unique().tolist())]
    techniques_software_df = techniques_software_df.sort_values(by='technique_ID', ascending=False).reset_index(drop=True)
    column_list = ['technique_ID','tactic_ID','software_ID','platform','group_ID','data_source_ID']
    for col in techniques_software_df.columns:
        if col in column_list:
            techniques_software_df[col] = format_cols_for_md(techniques_software_df[col])
        else:
           techniques_software_df[col] =  replace_chars(techniques_software_df[col])
    return techniques_software_df

In [25]:
def info_techniques_platforms(results_df, mitre_domain):
    '''
    Función encargada de filtrar mediante el df facilitado y formatear la tabla de información de plataformas. 
    '''
    techniques_platforms_df = get_data_from_relation_techniques_platforms(mitre_domain)
    techniques_platforms_df = techniques_platforms_df[techniques_platforms_df['technique_ID'].isin(results_df['technique ID'].unique().tolist())]
    techniques_platforms_df = techniques_platforms_df.sort_values(by='technique_ID', ascending=False).reset_index(drop=True)
    column_list = ['technique_ID','tactic_ID','platform_ID','platform','group_ID','data_source_ID']
    for col in techniques_platforms_df.columns:
        if col in column_list:
            techniques_platforms_df[col] = format_cols_for_md(techniques_platforms_df[col])
        else:
           techniques_platforms_df[col] =  replace_chars(techniques_platforms_df[col])
    return techniques_platforms_df

In [26]:
def info_techniques_groups(results_df, mitre_domain):
    '''
    Función encargada de filtrar mediante el df facilitado y formatear la tabla de información de grupos. 
    '''
    techniques_groups_df = get_data_from_relation_technique_groups(mitre_domain)
    techniques_groups_df = techniques_groups_df[techniques_groups_df['technique_ID'].isin(results_df['technique ID'].unique().tolist())]
    techniques_groups_df = techniques_groups_df.sort_values(by='technique_ID', ascending=False).reset_index(drop=True)
    column_list = ['technique_ID','tactic_ID','platform_ID','platform','group_ID','data_source_ID']
    for col in techniques_groups_df.columns:
        if col in column_list:
            techniques_groups_df[col] = format_cols_for_md(techniques_groups_df[col])
        else:
           techniques_groups_df[col] =  replace_chars(techniques_groups_df[col])
    return techniques_groups_df

In [27]:
def info_techniques_datasources(results_df, mitre_domain):
    '''
    Función encargada de filtrar mediante el df facilitado y formatear la tabla de información de datasources. 
    '''
    techniques_datasources_df = get_data_from_relation_techniques_datasources(mitre_domain)
    techniques_datasources_df = techniques_datasources_df[techniques_datasources_df['technique_ID'].isin(results_df['technique ID'].unique().tolist())]
    techniques_datasources_df = techniques_datasources_df.sort_values(by='technique_ID', ascending=False).reset_index(drop=True)
    column_list = ['technique_ID','tactic_ID','platform_ID','platform','group_ID','data_source_ID']
    for col in techniques_datasources_df.columns:
        if col in column_list:
            techniques_datasources_df[col] = format_cols_for_md(techniques_datasources_df[col])
        else:
           techniques_datasources_df[col] =  replace_chars(techniques_datasources_df[col])
    return techniques_datasources_df

In [28]:
def folder_list(ruta):
    '''
    Función encargada de listar las carpetas de una ruta dada.. 
    '''
    try:
        # Lista todas las carpetas en la ruta dada con sus rutas completas
        carpetas = [os.path.join(ruta, nombre) for nombre in os.listdir(ruta) if os.path.isdir(os.path.join(ruta, nombre))]
        return carpetas
    except FileNotFoundError:
        print(f"La ruta {ruta} no existe.")
        return []
    except PermissionError:
        print(f"No tienes permisos para acceder a la ruta {ruta}.")
        return []

In [29]:
def compress_subfolders_7z(ruta_base):
    '''
    Función encargada de comprimir como archivo .7z las carpetas generadas para los outputs que contienen las reglas de detección. 
    '''
    nombre_carpeta_base = os.path.basename(ruta_base.rstrip(os.sep))

    for subcarpeta in os.listdir(ruta_base):
        ruta_subcarpeta = os.path.join(ruta_base, subcarpeta)
        
        if os.path.isdir(ruta_subcarpeta):
            carpetas_a_comprimir = [os.path.join(ruta_subcarpeta, nombre) for nombre in os.listdir(ruta_subcarpeta) if os.path.isdir(os.path.join(ruta_subcarpeta, nombre))]
            
            if carpetas_a_comprimir:
                nombre_archivo = f"detection_rules_{nombre_carpeta_base}_{subcarpeta}.7z"
                archivo_salida = os.path.join(ruta_subcarpeta, nombre_archivo)
                # Importante que 7z esté en la path del sistema
                comando = ['7z', 'a', archivo_salida] + carpetas_a_comprimir
                
                try:
                    subprocess.run(comando, check=True)
                    print(f"Archivo {archivo_salida} creado exitosamente.")
                    
                    for carpeta in carpetas_a_comprimir:
                        shutil.rmtree(carpeta)
                        print(f"Carpeta {carpeta} eliminada.")
                    
                except subprocess.CalledProcessError as e:
                    print(f"Error al crear {archivo_salida}: {e}")
            else:
                print(f"No se encontraron carpetas en {ruta_subcarpeta} para comprimir.")

#### **Ejecución**

##### **Parámetros inciales**

In [30]:
mitre_domain = 'enterprise' # enterprise, mobile, ics
zip_and_delete = True # Crea un zip con las reglas evitando que obsidian pueda acceder a los ficheros con las reglas de detección
separator_item = ',' # 

In [31]:
# Parámetros NO modificables
rules_path = os.path.join(os.getcwd(), 'outputs', mitre_domain)
output_path = os.path.join(os.getcwd(), 'query_outputs', 'by_actor')
check_and_create_folder(output_path)
source_folders = [folder for folder in os.listdir(rules_path) if os.path.isdir(os.path.join(rules_path, folder))]

La ruta ya existe: c:\Users\jelopez\Documents\CyberProof\python\develop\get_rules_and_classify_by_ttp\query_outputs\by_actor


In [32]:
if len(folder_list(os.path.join(os.getcwd(),'query_outputs', 'by_actor'))) > 0:
    print('Antes de ejecutar, por favor retira los archivos generados previamente de la carpeta query_outputs/by_actor')

Antes de ejecutar, por favor retira los archivos generados previamente de la carpeta query_outputs/by_actor


In [33]:
# Obtenemos el listado de archivos de riesgos a ejecutar
input_files = list_files_in_directory(os.path.join(os.getcwd(),'input_actor'))

Se encontraron 2 archivo(s) CSV en la carpeta 'c:\Users\jelopez\Documents\CyberProof\python\develop\get_rules_and_classify_by_ttp\input_actor':
c:\Users\jelopez\Documents\CyberProof\python\develop\get_rules_and_classify_by_ttp\input_actor\mitre-ttps-recommended-actor.csv
c:\Users\jelopez\Documents\CyberProof\python\develop\get_rules_and_classify_by_ttp\input_actor\mitre-ttps-recommendedv2-actor.csv


##### **Cargamos las tablas MITRE de grupos y tácticas para cruzar con la tabla inputs**

In [34]:
info_groups_df,info_tactics_df = get_group_tactic_information(mitre_domain)

##### **Creamos el resumen de ficheros de riesgos analizados**

In [35]:
for input_file in input_files:
    print(os.path.splitext(os.path.basename(input_file))[0])

mitre-ttps-recommended-actor
mitre-ttps-recommendedv2-actor


In [36]:
if len(input_files) > 0:
    with open(os.path.join(os.getcwd(),'query_outputs', 'by_actor', 'resumen_riesgos.md'), 'w') as file:
        file.write(f"**Resumen de riesgos analizados y archivos asociados:**\n\n")
        for input_file in input_files:
            file.write(f"[[resumen_{os.path.splitext(os.path.basename(input_file))[0]}.md]]\n")
            file.write("\n")
        file.close()


##### **Ejecución principal: generación de md's y copiado de reglas**

In [37]:
for i in input_files:
    input_df = pd.read_csv(i, sep=separator_item, dtype=str)
    input_df = clean_input_df(input_df)

In [38]:
# Recorremos cada uno de los csv inputs
for i in input_files:
    input_df = pd.read_csv(i, sep=separator_item, dtype=str)
    # Formateamos especificamente para grupos
    input_df = clean_input_df(input_df)
    # Unimos la información input con la información MITRE
    union = pd.merge(input_df, info_groups_df, on='group', how='left')
    union = pd.merge(union, info_tactics_df, on='tactic', how='left')
    union['tactic'] = union['tactic'].str.upper()
    union['group'] = union['group'].str.upper()
    
    # eliminamos la tabla que no necesitamos en adelante para liberar memoria
    del input_df

    # generamos el resumen global del riesgo analizado
    file_name = os.path.splitext(os.path.basename(i))[0]
    resume_filename = f'resumen_{file_name}.md'
    resume_path = os.path.join(output_path, resume_filename)
    resume_ttps_by_group_tactic = union.groupby(['group', 'tactic'])['technique_id'].count().reset_index(name='techniques')

    with open(resume_path, 'w') as file:
        file.write(f"**Archivos relacionados por grupo para el riesgo {file_name}**\n\n")
        for group in union['group'].unique()[pd.notna(union['group'].unique())]:
            file.write(f"[[riesgos_{file_name}_{group.upper()}.md]]\n")
            file.write("\n")
        file.write("### Resumen técnicas por grupo y táctica:\n\n")
        file.write("| group | tactic | techniques |\n")
        file.write("| ------------ | ------ | ---- |\n")
        for _, row in resume_ttps_by_group_tactic.iterrows():
            file.write(f"| {row['group']} | {row['tactic']} | {row['techniques']} |\n")
        file.write("\n\n")
        file.close()
        
    # recorremos cada uno de los grupos para la construccion de la capa superior 
    for group in union['group'].unique()[pd.notna(union['group'].unique())]:
        file_group_filename = f'riesgos_{file_name}_{group.upper()}.md'
        file_group_path = os.path.join(output_path, file_name, file_group_filename)
        optional_folder = file_name.replace("\\", "-").replace("/", "-")

        # filtramos la tabla por el grupo en cuestión
        union_group_df = union[(union['group']==group.upper())]
        # Obtenemos la lista de técnicas filtradas para este grupo
        ttp_list = union_group_df['technique_id'].unique()[pd.notna(union_group_df['technique_id'].unique())]
        
        # filtramos el df de reglas de detección para las técnicas a consultar
        rules_df = get_rules_df(source_folders, ttp_list, rules_path)
        # copiamos las reglas de detección para dicho grupo y lista de ttp's
        copy_files(rules_df, output_path, optional_folder, group.upper())

        print('Proceso de copiado de reglas finalizado')

        # obtenemos la informacion mitre que usaremos para relacionar con las ttp's
        techniques_tactics_df = info_techniques_tactics(rules_df, mitre_domain)
        techniques_software_df = info_techniques_software(rules_df, mitre_domain)
        techniques_platforms_df = info_techniques_platforms(rules_df, mitre_domain)
        techniques_groups_df = info_techniques_groups(rules_df, mitre_domain)
        techniques_datasources_df = info_techniques_datasources(rules_df, mitre_domain)
        techniques_df = info_techniques(rules_df, mitre_domain)

        # unimos las tablas informativas
        info_df = pd.merge(techniques_tactics_df, techniques_software_df, on=['technique_ID', 'technique'], how='left')
        info_df = pd.merge(info_df, techniques_platforms_df, on=['technique_ID', 'technique'], how='left')
        info_df = pd.merge(info_df, techniques_groups_df, on=['technique_ID', 'technique'], how='left')
        info_df = pd.merge(info_df, techniques_datasources_df, on=['technique_ID', 'technique'], how='left')
        info_df = pd.merge(info_df, techniques_df, on=['technique_ID', 'technique'], how='left')

        # eliminamos tablas que no necesitamos en adelante para liberar memoria
        del techniques_tactics_df, techniques_software_df, techniques_platforms_df, techniques_groups_df, techniques_datasources_df, techniques_df

        info_df.columns = info_df.columns.str.replace('_', ' ', regex=False)

        rules_df = rules_df[['technique ID', 'source', 'rule']]
        resume_rules_df = rules_df.groupby(['technique ID', 'source'])['rule'].count().reset_index(name='rules')
        rules_technique_group = resume_rules_df.groupby(['technique ID'])['rules'].sum().reset_index(name='rules').query('`technique ID` in @ttp_list')

        # generamos los markdown tanto a nivel de grupo como de ttp
        generate_markdown(union_group_df, info_df, resume_rules_df, rules_df, os.path.join(output_path, optional_folder), group, rules_technique_group, optional_folder)
    # zipeamos las reglas de deteccion y eliminamos las carpetas
    if zip_and_delete:
        risks_forders = folder_list(output_path)
        for rf in risks_forders:
            compress_subfolders_7z(rf)
    print(f"-------------------- Finalizado {i} !------------------------------------------------------------------------------")
print('Proceso finalizado correctamente!')

Proceso de copiado de reglas finalizado
Proceso de generacion de archivos .md finalizado correctamente!
Proceso de copiado de reglas finalizado
Proceso de generacion de archivos .md finalizado correctamente!
Proceso de copiado de reglas finalizado
Proceso de generacion de archivos .md finalizado correctamente!
Proceso de copiado de reglas finalizado
Proceso de generacion de archivos .md finalizado correctamente!
Proceso de copiado de reglas finalizado
Proceso de generacion de archivos .md finalizado correctamente!
Proceso de copiado de reglas finalizado
Proceso de generacion de archivos .md finalizado correctamente!
Proceso de copiado de reglas finalizado
Proceso de generacion de archivos .md finalizado correctamente!
Proceso de copiado de reglas finalizado
Proceso de generacion de archivos .md finalizado correctamente!
Proceso de copiado de reglas finalizado
Proceso de generacion de archivos .md finalizado correctamente!
Proceso de copiado de reglas finalizado
Proceso de generacion de